In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
from db_config import DB_URL

engine = create_engine(DB_URL, pool_pre_ping=True)

In [ ]:
#  1. Load both course files
mat = pd.read_csv(r"c:\ExcelR\Project\student_dropout\data\student-mat.csv", sep=";")
por = pd.read_csv(r"c:\ExcelR\Project\student_dropout\data\student-por.csv", sep=";")

# Tag which course each row belongs to
mat["course"] = "math"
por["course"] = "por"

In [3]:
#  2. Merge into one DataFrame 
df = pd.concat([mat, por], ignore_index=True)
df.columns = df.columns.str.lower().str.strip()
print(f"Total students: {len(df):,}")
print(f"Columns: {df.columns.tolist()}")

Total students: 1,044
Columns: ['school', 'sex', 'age', 'address', 'famsize', 'pstatus', 'medu', 'fedu', 'mjob', 'fjob', 'reason', 'guardian', 'traveltime', 'studytime', 'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'dalc', 'walc', 'health', 'absences', 'g1', 'g2', 'g3', 'course']


In [5]:
# 3. Generate a unique student_id 
df["student_id"] = range(1, len(df) + 1)

In [6]:
#  4. Parse binary yes/no columns to 1/0 for SQL compatibility 
binary_cols = ["schoolsup","famsup","paid","activities",
               "nursery","higher","internet","romantic"]
for col in binary_cols:
    if col in df.columns:
        df[col] = df[col].map({"yes": 1, "no": 0})

In [7]:
#  5. Validate grade range — must be 0-20 
for col in ["g1", "g2", "g3"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df[col] = np.clip(df[col], 0, 20)

# Drop rows where final grade G3 is missing
df.dropna(subset=["g3"], inplace=True)

In [8]:
#  6. NumPy: compute GPA (avg of G1, G2, G3)
grade_matrix = df[["g1","g2","g3"]].values          # shape (N, 3)
df["gpa"]    = np.mean(grade_matrix, axis=1).round(2)

In [9]:
#  7. NumPy: grade trend (G3 to G1 = improvement or decline) 
df["grade_trend"] = np.round(df["g3"].values - df["g1"].values, 2)

In [10]:
# 8. Pass / fail flag (passing = G3 >= 10 on 0-20 scale) 
df["passed"] = np.where(df["g3"] >= 10, True, False)

print(f"\nPass rate     : {df['passed'].mean()*100:.1f}%")
print(f"Avg GPA       : {df['gpa'].mean():.2f}")
print(f"Avg absences  : {df['absences'].mean():.1f}")


Pass rate     : 78.0%
Avg GPA       : 11.27
Avg absences  : 4.4


In [11]:
#  9. Split into dim and fact tables 
dim_cols = [
    "student_id","school","sex","age","address","famsize",
    "pstatus","medu","fedu","mjob","fjob","internet",
    "famsup","paid","health","course"
]
fact_cols = [
    "student_id","course","studytime","failures",
    "absences","g1","g2","g3","gpa","grade_trend","passed"
]

dim_student  = df[dim_cols].copy()
fact_grades  = df[fact_cols].copy()

# Placeholder columns — filled in 03_risk_scoring.py
fact_grades["at_risk"]    = False
fact_grades["risk_score"] = 0.0

In [13]:
#  10. Load to PostgreSQL 
dim_student.to_sql("dim_student", engine,
                   if_exists="replace", index=False,
                   method="multi", chunksize=1000)

fact_grades.to_sql("fact_grades", engine,
                   if_exists="replace", index=False,
                   method="multi", chunksize=1000)

print(f"\ndim_student  loaded: {len(dim_student):,} rows")
print(f"fact_grades  loaded: {len(fact_grades):,} rows")

# Verify
with engine.connect() as conn:
    for table in ["dim_student","fact_grades"]:
        n = conn.execute(text(f"SELECT COUNT(*) FROM {table}")).scalar()
        print(f"  PostgreSQL {table}: {n:,} rows ✓")


dim_student  loaded: 1,044 rows
fact_grades  loaded: 1,044 rows
  PostgreSQL dim_student: 1,044 rows ✓
  PostgreSQL fact_grades: 1,044 rows ✓
